In [1]:
import json

def convert_to_v1(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    new_data = []

    for item in data:
        lines = item["input"].split("\n")

        # Get last User message
        last_user = None
        for line in reversed(lines):
            if line.startswith("User:"):
                last_user = line.replace("User:", "").strip()
                break

        if not last_user:
            continue

        # Convert output → list
        outputs = [o.strip() for o in item["output"].split("\n") if o.strip()]

        # Keep max 3
        outputs = outputs[:3]

        new_item = {
            "instruction": "Generate 3 smart reply suggestions",
            "input": f"Conversation:\nFriend: {last_user}",
            "output": outputs
        }

        new_data.append(new_item)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(new_data, f, indent=2, ensure_ascii=False)

    print("✅ Done!")


# ✅ USE THIS (based on your screenshot)
convert_to_v1(
    "making v1 dataset/old_dataset.json",
    "making v1 dataset/v1_dataset.json"
)

✅ Done!


# 16.4.25

In [3]:
import json

# ✅ FILE PATH
file_path = "making v1 dataset/old_dataset.json"

# ✅ LOAD DATA
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

clean_data = []

# =========================
# 🔧 CLEAN REPLIES
# =========================
def clean_replies(output):
    if isinstance(output, str):
        replies = output.split("\n")
    else:
        replies = output

    cleaned = []

    for r in replies:
        r = str(r).strip()

        # remove numbering like "1." "2." etc
        if r.startswith(("1.", "2.", "3.", "4.", "5.")):
            r = r[2:].strip()

        # skip bad/empty
        if len(r) < 3:
            continue

        cleaned.append(r)

    # remove duplicates + keep only 3
    return list(dict.fromkeys(cleaned))[:3]


# =========================
# 🔧 EXTRACT FRIEND MESSAGE
# =========================
def extract_friend_message(text):
    lines = text.split("\n")

    # Try to find last Friend message
    for line in reversed(lines):
        if "Friend:" in line:
            return line.replace("Friend:", "").strip()

    # fallback → take last line
    return lines[-1].strip() if lines else None


# =========================
# 🔄 PROCESS DATA
# =========================
for item in data:
    input_text = item.get("input", "")
    output = item.get("output", [])

    friend_msg = extract_friend_message(input_text)
    replies = clean_replies(output)

    # ✅ FIX: Relax filtering (main reason you got 0 samples)
    if not friend_msg:
        continue

    if len(replies) < 1:   # allow even 1 → later we ensure 3
        continue

    # ensure exactly 3 replies
    while len(replies) < 3:
        replies.append(replies[-1])

    replies = replies[:3]

    clean_item = {
        "instruction": "Generate 3 smart reply suggestions",
        "input": f"Conversation:\nFriend: {friend_msg}",
        "output": replies
    }

    clean_data.append(clean_item)


# =========================
# 💾 SAVE CLEAN DATASET
# =========================
output_path = "making v1 dataset/clean_v1_dataset.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(clean_data, f, indent=2, ensure_ascii=False)

print(f"✅ Done! Clean dataset saved with {len(clean_data)} samples")

✅ Done! Clean dataset saved with 3014 samples
